<a href="https://colab.research.google.com/github/SiddhantZade1/Inflation-Market-Study/blob/main/analysis_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inflation Regimes & Market Returns
### *Not all inflation is the same — and your hedge shouldn't be either.*

---

**Central Question:**  
Do traditional inflation hedges (gold, commodities, TIPS, REITs) actually protect investors —  
and does the answer change depending on the *type* of inflation environment?

**Approach:**
1. Classify 60+ years of US inflation into four distinct regimes using a rule-based classifier, validated with a Hidden Markov Model
2. Measure the real (inflation-adjusted) performance of 7 asset classes within each regime
3. Identify which equity sectors have the highest "inflation beta" and hedge effectiveness

**Key Finding Preview:**  
> Gold is widely considered an inflation hedge. The data shows it only reliably outperforms in *one* of four inflation regimes.

---

## 0. Setup

In [1]:
# Colab setup (run this first)

# 1) Clone your repo (edit the URL)
REPO_URL = "https://github.com/SiddhantZade1/Inflation-Market-Study.git"
REPO_DIR = REPO_URL.split("/")[-1].replace(".git","")

# Uncomment if you're running this notebook from Colab directly
!git clone {REPO_URL}
%cd {REPO_DIR}

# 2) Install dependencies (if you have requirements.txt)
!pip -q install -r requirements.txt

# 3) Optional: if you use local caching, create a data folder
# import os
# os.makedirs("data", exist_ok=True)


Cloning into 'Inflation-Market-Study'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 50 (delta 19), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 237.41 KiB | 2.05 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/Inflation-Market-Study
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.5/109.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 145.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 152.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.3/233.3 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 15.8 MB/s eta 0:00:00
   ━

In [2]:
# Standard library
import warnings
import sys
import os
warnings.filterwarnings('ignore')

# Third-party
import numpy as np
import pandas as pd
import plotly.io as pio

# Local
sys.path.append(os.getcwd())  # repo root if you %cd into it
from src import DataFetcher, RegimeClassifier, AssetClassAnalysis, SectorHedgeAnalysis

# Plotly default renderer
pio.renderers.default = 'colab'

# Display settings
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 40)

print('Setup complete ✓')

Setup complete ✓


In [3]:
# ─── CONFIGURATION ────────────────────────────────────────────────────────────
# FRED API key:
# - In Colab: you can store it in "Secrets" as FRED_API_KEY, then we read it via google.colab.userdata
# - Or set an environment variable: export FRED_API_KEY=...
# - Or paste it when prompted (recommended: don't commit keys to git)

import os
from getpass import getpass

FRED_API_KEY = os.environ.get("FRED_API_KEY")

try:
    from google.colab import userdata
    FRED_API_KEY = FRED_API_KEY or userdata.get("FRED_API_KEY")
except Exception:
    pass

if not FRED_API_KEY or FRED_API_KEY.strip() in {"", "YOUR_KEY_HERE"}:
    FRED_API_KEY = getpass("Enter FRED_API_KEY (input hidden): ").strip()

START_DATE    = '1962-01-01'
FORCE_REFRESH = False


Enter FRED_API_KEY (input hidden): ··········


---
## 1. Data
Pulling macro series from FRED and asset/sector prices from Yahoo Finance.  
All data is cached locally after the first download.

In [4]:
fetcher = DataFetcher(
    fred_api_key  = FRED_API_KEY,
    start_date    = START_DATE,
    force_refresh = FORCE_REFRESH,
)

macro   = fetcher.get_macro()
assets  = fetcher.get_assets()
sectors = fetcher.get_sectors()
master  = fetcher.get_master()

print(f"Macro data    : {macro.shape[0]:>5} months | {macro.shape[1]:>3} columns | {macro.index.min().date()} → {macro.index.max().date()}")
print(f"Asset prices  : {assets['prices'].shape[0]:>5} months | {assets['prices'].shape[1]:>3} tickers")
print(f"Sector prices : {sectors['prices'].shape[0]:>5} months | {sectors['prices'].shape[1]:>3} tickers")
print(f"Master dataset: {master.shape[0]:>5} months | {master.shape[1]:>3} columns")

Macro data    :   769 months |  28 columns | 1962-01-31 → 2026-01-31
Asset prices  :   398 months |   8 tickers
Sector prices :   327 months |  10 tickers
Master dataset:   769 months |  46 columns


In [5]:
# Quick data quality check
fetcher.data_summary()

,file,start,end,rows,nulls_pct
0,asset_prices.csv,1993-01-31,2026-02-28,398,30.7%
1,asset_returns.csv,1993-01-31,2026-02-28,398,31.0%
2,macro.csv,1962-01-31,2026-01-31,769,8.0%
3,master.csv,1962-01-31,2026-01-31,769,29.2%
4,sector_prices.csv,1998-12-31,2026-02-28,327,6.2%
5,sector_returns.csv,1998-12-31,2026-02-28,327,6.5%


In [6]:
# CPI overview
macro[['cpi_yoy', 'core_cpi_yoy', 'pce_yoy', 'cpi_3m_ann', 'breakeven_10y']].tail(12).round(2)

,cpi_yoy,core_cpi_yoy,pce_yoy,cpi_3m_ann,breakeven_10y
date,,,,,
2025-02-28,2.80,3.14,2.71,4.04,2.42
2025-03-31,2.38,2.81,2.36,2.78,2.33
2025-04-30,2.33,2.78,2.28,1.69,2.24
2025-05-31,2.38,2.77,2.46,1.18,2.31
2025-06-30,2.68,2.91,2.59,2.08,2.30
2025-07-31,2.74,3.05,2.61,2.35,2.38
2025-08-31,2.94,3.11,2.75,3.37,2.38
2025-09-30,3.02,3.02,2.79,3.54,2.37
2025-10-31,2.73,2.70,2.72,NaN,2.31


---
## 2. Inflation Regime Classification

We define four regimes based on the **level** and **momentum** of CPI YoY:

| Regime | Level | Momentum |
|--------|-------|----------|
| **Low & Stable** | < 2.5% | Flat |
| **Rising** | Any | Accelerating |
| **High & Stable** | > 4.0% | Flat |
| **Falling** | > 2.5% | Decelerating |

Momentum = 6-month change in CPI YoY. This matters because the *direction* of inflation often drives asset prices more than the absolute level.

In [7]:
rc = RegimeClassifier(macro)
macro_regimes = rc.classify_threshold(
    low_threshold      = 2.5,
    high_threshold     = 4.0,
    momentum_window    = 6,
    momentum_threshold = 0.3,
)

# Distribution
print("Regime distribution:")
print(macro_regimes['regime_label'].value_counts())
print(f"\nTotal months classified: {macro_regimes['regime_label'].notna().sum()}")

Regime distribution:
regime_label
Rising           291
Low & Stable     280
Falling          166
High & Stable     32
Name: count, dtype: int64

Total months classified: 769


In [8]:
# Visualise regimes on CPI history
fig = rc.plot_regimes(df=macro_regimes)
fig.show()

In [9]:
# Summary statistics per regime
regime_stats = RegimeClassifier.regime_stats(macro_regimes)
# Show key columns cleanly
regime_stats.xs('mean', level=1, axis=1)[['cpi_yoy', 'core_cpi_yoy', 'fed_funds', 'real_fed_funds', 'yield_curve_slope']].round(2)

,cpi_yoy,core_cpi_yoy,fed_funds,real_fed_funds,yield_curve_slope
regime_label,,,,,
Falling,5.14,5.49,6.82,1.68,0.51
High & Stable,5.63,5.70,7.76,2.12,0.44
Low & Stable,1.95,2.35,3.17,1.25,1.09
Rising,4.70,4.02,5.04,0.35,0.82


### 2b. HMM Validation
We validate the threshold classification with a data-driven Hidden Markov Model.  
If the HMM finds similar regime boundaries, it increases confidence in our taxonomy.

In [10]:
!pip install hmmlearn

macro_hmm = rc.classify_hmm(n_states=4, features=['cpi_yoy', 'cpi_3m_ann', 'real_fed_funds'])

# Agreement between threshold and HMM
common = macro_regimes.join(macro_hmm[['hmm_regime_label']], how='inner').dropna()

from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(common['regime_label'], common['hmm_regime_label'])
print(f"Adjusted Rand Index (threshold vs HMM): {ari:.3f}")
print("  (1.0 = perfect agreement, 0.0 = random, negative = worse than random)")

# Cross-tabulation
pd.crosstab(common['regime_label'], common['hmm_regime_label'], normalize='index').round(2)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 6.7 MB/s eta 0:00:00
Adjusted Rand Index (threshold vs HMM): 0.127
  (1.0 = perfect agreement, 0.0 = random, negative = worse than random)


hmm_regime_label,High & Stable,Low & Stable,Rising
regime_label,,,
Falling,0.31,0.14,0.55
High & Stable,0.67,0.00,0.33
Low & Stable,0.05,0.73,0.22
Rising,0.26,0.65,0.09


---
## 3. Asset Class Performance by Regime

Now the central question: which asset classes protect purchasing power  
in each inflation regime — and which are commonly believed to but don't?

**Assets:**
- SPY  → US Equities (S&P 500)
- IEF  → Long Treasuries
- TIP  → TIPS (inflation-linked bonds)
- GLD  → Gold
- VNQ  → REITs
- DJP  → Commodities
- BIL  → Cash / T-Bills

*Note: ETF data starts ~2000–2007. Regime labels are from the threshold classifier.*

In [11]:
# Merge regimes into master dataset
master_with_regimes = master.join(
    macro_regimes[['regime_label', 'regime_id', 'cpi_momentum']],
    how='left'
)

aca = AssetClassAnalysis(
    master_df  = master_with_regimes,
    regime_col = 'regime_label',
)

summary = aca.regime_returns_summary()
print(f"Summary table: {summary.shape}")
summary.head(12)

Summary table: (32, 10)


,regime,asset,n_months,mean_ret,median_ret,std_ret,sharpe,pct_positive,best_month,worst_month
0,Low & Stable,T-Bills (Cash),107,0.10,0.02,0.16,2.09,56.07,0.46,-0.23
1,Low & Stable,Commodities,110,-0.07,-0.11,5.01,-0.05,49.09,13.84,-21.52
2,Low & Stable,Gold,118,1.04,1.03,4.71,0.76,55.93,12.57,-16.14
3,Low & Stable,Treasuries 7-10yr,132,0.52,0.35,1.96,0.90,59.85,7.75,-5.02
4,Low & Stable,Short Treasuries,132,0.21,0.16,0.40,1.79,71.97,1.24,-1.06
5,Low & Stable,US Equities,207,1.17,1.80,4.46,0.91,68.60,12.70,-16.52
6,Low & Stable,TIPS,122,0.34,0.47,1.78,0.64,64.75,6.50,-8.09
7,Low & Stable,REITs,118,0.68,1.73,7.45,0.31,61.02,30.68,-31.73
8,Rising,T-Bills (Cash),91,0.06,0.00,0.12,1.66,50.55,0.51,-0.04
9,Rising,Commodities,94,0.52,1.60,5.46,0.33,61.70,12.16,-15.63


In [12]:
# Mean monthly returns by asset & regime — the key chart
fig = aca.plot_regime_returns(metric='mean_ret')
fig.show()

In [13]:
# Sharpe ratios — risk-adjusted view
fig = aca.plot_regime_returns(metric='sharpe')
fig.show()

In [14]:
# Heatmap: median monthly return (easy-to-scan summary)
asset_return_cols = [c for c in master_with_regimes.columns if c.endswith('_ret') and not c.endswith('_sec_ret')]
asset_returns_df  = master_with_regimes[asset_return_cols].copy()

fig = rc.plot_regime_heatmap(
    returns_df = asset_returns_df,
    df         = master_with_regimes,
    regime_col = 'regime_label',
)
fig.show()

In [15]:
# Real (inflation-adjusted) returns — the honest view
real_summary = aca.real_returns_summary(cpi_mom_col='cpi_mom')

# Pivot for readability
real_pivot = real_summary.pivot(index='asset', columns='regime', values='annualised')
real_pivot = real_pivot[[c for c in ['Low & Stable', 'Rising', 'High & Stable', 'Falling'] if c in real_pivot.columns]]
print("Annualised Real Returns by Regime (%):\n")
real_pivot.round(2)

Annualised Real Returns by Regime (%):



regime,Low & Stable,Rising,High & Stable,Falling
asset,,,,
Commodities,-2.16,2.45,7.85,-15.84
Gold,11.30,8.02,-0.33,10.48
REITs,6.91,7.03,-14.16,8.89
Short Treasuries,1.30,-2.28,-6.74,-1.27
T-Bills (Cash),0.02,-2.95,-2.17,0.67
TIPS,2.79,1.33,-9.14,-5.72
Treasuries 7-10yr,4.96,-0.60,-20.57,-5.69
US Equities,12.47,4.25,-13.29,7.55


In [16]:
# Nominal vs real comparison
fig = aca.plot_real_vs_nominal()
fig.show()

In [17]:
# Cumulative returns (full period, log scale)
fig = aca.plot_cumulative()
fig.show()

### Observations

*(Fill these in after running the analysis — these are the analytical insights  
that will anchor your README and make the project memorable.)*

- **Gold**: Strong in `_____` regime, weak in `_____`
- **Commodities**: The best performer in `_____` but hurt real returns in `_____`
- **TIPS**: Designed for inflation protection — does the data agree? In which regimes?
- **Equities**: Widely held but often misunderstood as an inflation hedge — what's the actual picture?
- **Cash**: Underrated in which regime?

---

## 4. Sector-Level Hedging Analysis

Among equity investors, which *sectors* are the most effective inflation hedges?  
We use three metrics:

1. **Rolling 24m correlation** — tracks the relationship dynamically over time
2. **Inflation beta** (OLS) — quantifies pass-through from CPI to sector returns
3. **Composite hedge score** — Sharpe + correlation + win-rate during high-inflation regimes

In [18]:
sha = SectorHedgeAnalysis(
    master_df     = master_with_regimes,
    regime_col    = 'regime_label',
    inflation_col = 'cpi_yoy',
    surprise_col  = 'inflation_surprise',
)

print('SectorHedgeAnalysis initialised ✓')

SectorHedgeAnalysis initialised ✓


In [19]:
# Rolling correlations — which sectors move with inflation?
fig = sha.plot_rolling_correlations(window=24)
fig.show()

In [20]:
# Inflation betas
betas = sha.inflation_betas(by_regime=False)
print(betas.sort_values('beta', ascending=False).to_string(index=False))

                sector  beta  r_squared  p_value
                Energy  3.54       0.02     0.01
            Financials  1.39       0.00     0.22
             Materials  0.98       0.00     0.38
           Health Care  0.51       0.00     0.50
            Technology  0.50       0.00     0.68
           Industrials  0.37       0.00     0.72
             Utilities  0.33       0.00     0.69
      Consumer Staples -0.07       0.00     0.92
           Real Estate -0.13       0.00     0.94
Consumer Discretionary -0.81       0.00     0.44


In [21]:
fig = sha.plot_inflation_betas()
fig.show()

In [22]:
# Regime betas — does the relationship change by inflation environment?
regime_betas = sha.inflation_betas(by_regime=True)
regime_betas.pivot(index='sector', columns='regime', values='beta').round(3)

regime,Falling,High & Stable,Low & Stable,Rising
sector,,,,
Consumer Discretionary,0.98,NaN,1.18,-2.76
Consumer Staples,0.60,NaN,0.23,-0.49
Energy,8.11,NaN,1.92,3.97
Financials,7.35,NaN,4.02,-1.45
Health Care,-0.31,NaN,0.72,1.05
Industrials,2.68,NaN,1.90,-1.23
Materials,2.92,NaN,2.65,-0.18
Real Estate,NaN,NaN,-2.05,0.81
Technology,-2.73,NaN,2.01,-0.81


In [23]:
# Composite hedge score
scores = sha.hedge_effectiveness()
print("Sector Hedge Effectiveness Scorecard:")
print(scores[['sector', 'sharpe_high_inf', 'corr_with_cpi', 'pct_pos_high_inf', 'hedge_score']].to_string(index=False))

Sector Hedge Effectiveness Scorecard:
                sector  sharpe_high_inf  corr_with_cpi  pct_pos_high_inf  hedge_score
                Energy             0.79           0.03             63.30         0.97
             Utilities             0.72          -0.02             64.70         0.89
      Consumer Staples             0.39          -0.05             59.70         0.57
            Technology             0.46          -0.12             60.40         0.51
           Health Care             0.42          -0.09             58.30         0.48
           Real Estate             0.42          -0.11             58.20         0.46
           Industrials             0.42          -0.10             57.60         0.45
             Materials             0.22          -0.12             55.40         0.28
Consumer Discretionary             0.21          -0.17             51.80         0.12
            Financials             0.05          -0.11             49.60         0.09


In [24]:
fig = sha.plot_hedge_scorecard()
fig.show()

In [25]:
# Sector × Regime heatmap — the full picture
fig = sha.plot_sector_regime_heatmap()
fig.show()

---
## 5. Summary & Key Takeaways

*(Complete after running the full analysis)*

### What the data shows

**On inflation regimes:**  
- Over the study period (XXXX–XXXX), the US spent X% of months in Low & Stable, X% Rising, X% High & Stable, X% Falling.
- The 2021–2022 inflation episode was primarily a `Rising` regime, transitioning to `High & Stable` in early 2022.

**On asset class hedging:**  
- The best single inflation hedge across all regimes was: ...
- TIPS performed as expected only in ...
- Gold's inflation-hedging reputation is primarily driven by ...

**On sector hedging:**  
- Energy had the highest inflation beta (β = X.XX), confirming its role as a commodity pass-through sector
- Technology had the lowest/most negative beta during High & Stable regimes (duration risk)
- The top-scoring sectors on the composite hedge scorecard were ...

### Limitations
- ETF-based data limits the equity/bond analysis to post-~2000
- Macro data (FRED) extends to 1960 for regime classification but can't be matched to sector returns pre-1998
- Regime boundaries are sensitive to threshold parameters — the HMM validation increases but doesn't guarantee robustness
- All returns are pre-tax and don't account for transaction costs or rebalancing
- US-only; international comparisons are a natural extension

---

## Appendix: Data Sources

| Series | Source | ID |
|--------|--------|----|
| CPI All Items | FRED | CPIAUCSL |
| CPI Core (ex Food & Energy) | FRED | CPILFESL |
| PCE Price Index | FRED | PCEPI |
| 10-Year Breakeven Inflation | FRED | T10YIE |
| Fed Funds Rate | FRED | FEDFUNDS |
| Asset/Sector Prices | Yahoo Finance | Various ETFs |

Full series catalogue: see `src/data_fetch.py`